# T5 Fine-tuning (Abstractive Summarization: Description -> Headline)

T5 is **encoder-decoder**. The encoder reads the Reuters description in both directions, the decoder
writes a headline one token at a time while attending back to that encoding. Classification models
like ModernBERT cannot emit text at all, and decoder-only models like GPT-2 have to treat the input as
a prefix to continue; the seq2seq split keeps "understand this" and "write that" as separate jobs.

T5 was pretrained on many tasks at once, so it needs a **task prefix** to know which one you want.
That is what `data_args.source_prefix` (`"summarize: "`) is for, and `headlines/t5/dataset.py`
prepends it to every input.

> **Before you run this:** switch on the GPU with **Runtime -> Change runtime type -> Hardware
> accelerator -> GPU**.

> **This one is slow.** `t5-base` over ~26,000 training rows for 2 epochs, with beam-search generation
> at every evaluation, runs for hours on a Colab GPU — long enough that the runtime may disconnect
> first. To get a full pass end to end in minutes instead, swap the arguments in the cell below for:
>
> ```python
> model_args = SummarizationModelArguments(model_name_or_path="google-t5/t5-small")
> training_args = seq2seq_arguments(SUMMARIZATION, output_dir="/content/artifacts/t5", num_train_epochs=1)
> ```

In [ ]:
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

## Install the package from the clone

The install has to be **editable** (`-e`). `headlines/config.py` computes

```python
PROJECT_ROOT = Path(__file__).resolve().parents[2]
```

so `REUTERS_PATH` is found relative to wherever `config.py` physically lives. An editable install
leaves it inside the clone next to `data/`; a regular install copies it into `site-packages`, where
`parents[2]` points at nothing useful and the default `data_path` breaks.

In [ ]:
%pip install -q -e /content/Bert-T5-GPT2

## Fetch the WordNet corpora

METEOR matches stems and synonyms, which needs WordNet. `_ensure_wordnet()` in
`headlines/t5/metrics.py` tries this download itself but swallows any failure, so METEOR would quietly
score lower instead of erroring. Doing it up front removes the ambiguity.

In [ ]:
import nltk


nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

## Confirm the GPU is visible

`resolve_precision()` in `headlines/utils.py` reads exactly these two values. bf16 where the
device supports it, fp32 everywhere else — never fp16. ModernBERT and T5 were both pretrained in
bf16 and emit NaN losses under fp16, which does not raise: training runs to completion and the score
comes back at zero. A T4 has no bf16, so it trains in fp32 and takes longer.

In [ ]:
import torch


print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device:        {torch.cuda.get_device_name(0)}")
    print(f"bf16 support:  {torch.cuda.is_bf16_supported()}")

## Configure the run

Configuration is split three ways, the same split the `transformers` example scripts use.
`SummarizationModelArguments` says which checkpoint to fine-tune, `SummarizationDataArguments` says
what to train on, and `SUMMARIZATION` in `headlines/t5/config.py` is a plain dictionary of training
settings that `seq2seq_arguments()` turns into a real `Seq2SeqTrainingArguments` — so anything the
library accepts is accepted here too.

Note the data arguments carry **two** length budgets rather than the single one the other pipelines
use: descriptions are long, headlines are short, and treating them the same would either truncate the
input or waste a lot of padding on the target.

In [ ]:
from headlines.config import seq2seq_arguments
from headlines.t5.config import SUMMARIZATION, SummarizationDataArguments, SummarizationModelArguments


model_args = SummarizationModelArguments()
data_args = SummarizationDataArguments(max_eval_samples=500)
training_args = seq2seq_arguments(
    SUMMARIZATION,
    output_dir="/content/artifacts/t5",
    generation_num_beams=1,
    per_device_eval_batch_size=32,
)

print(model_args)
print(data_args)

## Load the Reuters descriptions

Each row pairs the description the model reads with the headline it is being asked to produce. Read
one pair in full before training anything — it is the clearest statement of the task.

In [ ]:
from headlines.data import load_csv


frame = load_csv(data_args.data_path, [data_args.source_column, data_args.target_column])
print(f"{len(frame):,} description/headline pairs after cleaning")
print()
print("Description:", frame.loc[0, data_args.source_column])
print()
print("Headline:   ", frame.loc[0, data_args.target_column])

## Tokenize source and target

`build_datasets` tokenizes the two sides in **separate calls**, because a single call would apply
`data_args.max_source_length` to both and leave the headlines padded out to 256 tokens. The targets go
through `text_target=`, which selects the tokenizer's target mode, and land in the `labels` column.

In [ ]:
from transformers import AutoTokenizer

from headlines.t5.dataset import build_datasets


tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)
datasets = build_datasets(data_args, tokenizer, training_args.seed)

for name, split in datasets.items():
    print(f"{name:<11} {len(split):>7,} rows  columns={split.column_names}")

example = datasets["train"][0]
print()
print("input: ", tokenizer.decode(example["input_ids"], skip_special_tokens=True)[:200], "...")
print("label: ", tokenizer.decode(example["labels"], skip_special_tokens=True))

## Build the Seq2SeqTrainer

`seq2seq_arguments()` builds a `Seq2SeqTrainingArguments` from the `SUMMARIZATION` dictionary, which
changes only the defaults: `predict_with_generate=True` makes evaluation actually generate text
instead of scoring teacher-forced logits, and `metric_for_best_model="rougeL"` picks the best
checkpoint on the metric the task is actually judged by. Keywords passed alongside the dictionary win
over it, so every one is still overridable per run.

Generation is what makes evaluation expensive. At the default `num_beams=4` over the full validation
split, the per-epoch eval costs more than the epoch of training it follows. The settings above cut
that down: `generation_num_beams=1` decodes greedily, and `max_eval_samples=500` on the data
arguments hands `apply_sample_limits` a 500-row validation slice. Both only affect *which checkpoint
gets picked*. The number quoted at the end comes from a full beam-4 pass over the held-out test
split, which nothing here has touched, so the reported score is not softened.

In [ ]:
from headlines.t5.train import build_trainer


trainer = build_trainer(model_args, data_args, training_args)

print(f"training on {len(trainer.train_dataset):,} rows for {training_args.num_train_epochs:g} epochs")
print(f"selecting checkpoints on {len(trainer.eval_dataset):,} validation rows (greedy)")
print(f"scoring once on {len(trainer.test_dataset):,} held-out test rows (beam 4)")
print(f"precision: fp16={trainer.args.fp16} bf16={trainer.args.bf16}")

## Smoke test before the real run

Finding out at minute twenty that a collator or a metric argument is wrong is a bad way to spend an
afternoon. This builds a throwaway `Trainer` that runs twenty steps and stops.

Every override goes through `build_trainer`, which merges as `**{**defaults, **arg_overrides}` — so
a caller can replace any key, including ones `run.py` sets itself. Passing them as literal keyword
arguments instead would raise `TypeError` on the collision.

In [ ]:
import gc

import torch


smoke_args = seq2seq_arguments(
    SUMMARIZATION,
    output_dir="/content/smoke",
    max_steps=20,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=10,
    save_steps=10,
    logging_steps=5,
    generation_num_beams=1,
)
smoke = build_trainer(model_args, data_args, smoke_args)
smoke.train()

del smoke
gc.collect()
torch.cuda.empty_cache()

## Fine-tune

In [ ]:
trainer.train()

## Learning curves

Loss and ROUGE-L do not always move together. Loss measures next-token agreement under teacher
forcing; ROUGE-L measures overlap in text the model generated on its own. Watching both is how you
notice a model that is getting more confident without getting more useful.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


history = pd.DataFrame(trainer.state.log_history)
steps = history.dropna(subset=["loss"])
evals = history.dropna(subset=["eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps["epoch"], steps["loss"], label="train", alpha=0.6)
axes[0].plot(evals["epoch"], evals["eval_loss"], marker="o", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(evals["epoch"], evals["eval_rougeL"], marker="o", color="darkorange")
axes[1].set_title("Validation ROUGE-L")
axes[1].set_xlabel("epoch")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.show()

## Score the held-out test split

Checkpoint selection ran greedy on a 500-row validation slice to keep the epochs moving. This does
not: `num_beams` and `max_length` are passed explicitly, so the reported ROUGE comes from the full
test split decoded with beam search. It is the slowest cell in the notebook and it runs once. Add
`.select(range(200))` to the dataset if you want a rough read instead.

`trainer.test_dataset` is the split `build_trainer` holds back. Early stopping never read it, so
unlike the validation numbers above this is a clean held-out estimate.

`Trainer.predict` returns predictions, label ids, and the metrics `compute_metrics` produced.

In [ ]:
prediction = trainer.predict(
    trainer.test_dataset,
    num_beams=4,
    max_length=data_args.max_target_length,
    metric_key_prefix="test",
)

for name, value in prediction.metrics.items():
    if name.startswith("test_") and isinstance(value, float):
        print(f"{name:<20} {value:.4f}")

### Metric bar chart

ROUGE-1 counts shared unigrams, ROUGE-2 shared bigrams, ROUGE-L the longest common subsequence.
ROUGE-2 is always the harshest of the three because word order has to match. METEOR sits apart from
all of them: it credits stems and synonyms, so a headline that paraphrases correctly scores here even
when ROUGE says it missed.

In [ ]:
names = ["rouge1", "rouge2", "rougeL", "meteor"]
values = [prediction.metrics[f"test_{name}"] for name in names]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, values, color=["steelblue", "steelblue", "steelblue", "darkorange"])
ax.bar_label(bars, fmt="%.4f")
ax.set_ylim(0, max(values) * 1.25)
ax.set_title("Held-out test split summarization scores")
ax.grid(True, axis="y", alpha=0.3)
plt.show()

print(f"mean generated length: {prediction.metrics['test_gen_len']:.2f} tokens")

### Eyeball a few predictions

The scores above cannot tell you whether a summary is wrong in an interesting way. Read the pairs.

In [ ]:
import numpy as np


generated = tokenizer.batch_decode(prediction.predictions, skip_special_tokens=True)
labels = np.where(prediction.label_ids != -100, prediction.label_ids, tokenizer.pad_token_id)
actual = tokenizer.batch_decode(labels, skip_special_tokens=True)

pd.set_option("display.max_colwidth", 80)
pd.DataFrame({"Generated": generated[:10], "Actual": actual[:10]})

### Summarize a new description

The prefix and the length cap come from `data_args` and the beam count matches the scoring cell above,
so this behaves exactly like evaluation did rather than quietly using different settings.

In [ ]:
description = (
    "The central bank held interest rates steady on Thursday, saying inflation had cooled enough to "
    "pause a tightening cycle that began eighteen months ago, but warned that policymakers were "
    "prepared to resume increases if price growth reaccelerated."
)

model = trainer.model
inputs = tokenizer(
    data_args.source_prefix + description,
    return_tensors="pt",
    truncation=True,
    max_length=data_args.max_source_length,
).to(model.device)

generated_ids = model.generate(**inputs, num_beams=4, max_length=data_args.max_target_length)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

## Save and download the checkpoint

`artifacts/` is gitignored, and a Colab runtime takes its disk with it when it disconnects. Download
the archive, or mount Drive and copy it there, before you close the tab.

In [ ]:
import shutil

from google.colab import files


trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

archive = shutil.make_archive("/content/t5-headlines", "zip", training_args.output_dir)
files.download(archive)